## 03 — Device Session Logs (semi-structured JSON, intentional schema drift)
#Generates app-session events with nested `geo` and `device_meta` structs.
#Writes one JSON file per day to simulate realistic batch drops, and
#intentionally varies the schema across a subset of files (missing fields,
#extra fields) so Bronze/Silver schema evolution has something real to do.

In [ ]:
dbutils.widgets.text("catalog", "northstar_dev")
dbutils.widgets.text("schema", "raw")
dbutils.widgets.text("volume", "landing")
dbutils.widgets.text("num_events", "260000")
dbutils.widgets.text("months_of_history", "15")
dbutils.widgets.text("random_seed", "42")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
VOLUME = dbutils.widgets.get("volume")
NUM_EVENTS = int(dbutils.widgets.get("num_events"))
MONTHS_HISTORY = int(dbutils.widgets.get("months_of_history"))
SEED = int(dbutils.widgets.get("random_seed"))

VOLUME_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
LOGS_PATH = f"{VOLUME_ROOT}/device_session_logs"

print(f"Target events: {NUM_EVENTS:,} | history: {MONTHS_HISTORY} months")

In [ ]:
customers_ref = spark.table(f"{CATALOG}.datagen_scratch.customer_ids")
customers_pdf = customers_ref.toPandas()
print(f"Customer pool: {len(customers_pdf):,}")

## Build events as Python dicts, day by day
Driver-side generation: 200k-300k nested JSON records is small enough that
distributing it adds more complexity than it saves, and writing day-by-day
files is inherently sequential (one output file per simulated daily drop)
so a Spark job wouldn't parallelize this step well anyway.

In [ ]:
from faker import Faker
import numpy as np
import json
import uuid
from datetime import datetime, timedelta

fake = Faker()
Faker.seed(SEED)
rng = np.random.default_rng(SEED + 5000)

EVENT_TYPES = ["login", "app_open", "card_freeze", "dispute_filed"]
EVENT_WEIGHTS = [0.45, 0.45, 0.06, 0.04]

OS_POOL = ["iOS", "Android"]
APP_VERSIONS = ["4.2.0", "4.2.1", "4.3.0", "4.3.1", "4.4.0", "5.0.0"]
DEVICE_MODELS = ["iPhone 14", "iPhone 15", "iPhone 16", "Pixel 8", "Pixel 9",
                  "Galaxy S23", "Galaxy S24", "OnePlus 12"]

CITY_GEO = [
    ("New York", 40.7128, -74.0060), ("London", 51.5074, -0.1278),
    ("Toronto", 43.6532, -79.3832), ("Berlin", 52.5200, 13.4050),
    ("Paris", 48.8566, 2.3522), ("Sydney", -33.8688, 151.2093),
    ("Singapore", 1.3521, 103.8198), ("Amsterdam", 52.3676, 4.9041),
    ("Dublin", 53.3498, -6.2603), ("Mexico City", 19.4326, -99.1332),
]

END_DATE = datetime.now()
START_DATE = END_DATE - timedelta(days=MONTHS_HISTORY * 30)
total_days = (END_DATE - START_DATE).days

customer_ids = customers_pdf["customer_id"].tolist()
n_cust = len(customer_ids)

# Each customer gets 1-3 stable device_ids reused across their sessions,
# rather than a fresh device per event -- more realistic than pure randomness.
customer_devices = {
    cid: [str(uuid.uuid4())[:12] for _ in range(int(rng.integers(1, 4)))]
    for cid in customer_ids
}

events_per_day = NUM_EVENTS // max(total_days, 1)

## Generate and write one file per simulated day
Schema drift injected deliberately on ~15% of days:
  - "legacy" days: device_meta missing `app_version` (older client behavior)
  - "enriched" days: geo struct gains an extra `accuracy_m` field
This is the kind of drift Autoloader's schema evolution / addNewColumns
setting is meant to absorb, and Silver flattening needs to tolerate.

In [ ]:
def make_event(day_offset, day_rng):
    cid = customer_ids[int(day_rng.integers(0, n_cust))]
    device_id = day_rng.choice(customer_devices[cid])
    event_type = str(day_rng.choice(EVENT_TYPES, p=EVENT_WEIGHTS))

    event_dt = START_DATE + timedelta(
        days=day_offset,
        seconds=int(day_rng.integers(0, 86400))
    )

    city, lat, lon = CITY_GEO[int(day_rng.integers(0, len(CITY_GEO)))]
    # small jitter so not every event in a city lands on the exact same point
    lat_j = round(lat + day_rng.normal(0, 0.05), 6)
    lon_j = round(lon + day_rng.normal(0, 0.05), 6)

    geo = {"lat": lat_j, "long": lon_j, "city": city}

    device_meta = {
        "os": str(day_rng.choice(OS_POOL)),
        "app_version": str(day_rng.choice(APP_VERSIONS)),
        "device_model": str(day_rng.choice(DEVICE_MODELS)),
    }

    return {
        "session_id": str(uuid.uuid4()),
        "customer_id": cid,
        "device_id": device_id,
        "event_type": event_type,
        "event_timestamp": event_dt.strftime("%Y-%m-%dT%H:%M:%S"),
        "ip_address": fake.ipv4_public(),
        "geo": geo,
        "device_meta": device_meta,
    }


# dbutils.fs.mkdirs(LOGS_PATH)
total_written = 0
files_written = 0

for day_offset in range(total_days):
    day_date = (START_DATE + timedelta(days=day_offset)).date()
    day_rng = np.random.default_rng(SEED + 9000 + day_offset)

    n_today = int(events_per_day * day_rng.uniform(0.6, 1.4))
    drift_roll = day_rng.random()

    day_events = []
    for _ in range(n_today):
        ev = make_event(day_offset, day_rng)

        if drift_roll < 0.10:
            del ev["device_meta"]["app_version"]
        elif drift_roll < 0.18:
            ev["geo"]["accuracy_m"] = round(float(day_rng.uniform(3, 50)), 1)

        day_events.append(ev)

    dbfs_target = f"{LOGS_PATH}/{day_date.strftime('%Y')}/{day_date.strftime('%m')}/part-{day_date}-00000.json"
    payload = "".join(json.dumps(ev) + "\n" for ev in day_events)
    dbutils.fs.put(dbfs_target, payload, overwrite=True)

    total_written += len(day_events)
    files_written += 1

print(f"Wrote {files_written} daily files, {total_written:,} total events to {LOGS_PATH}")

## Spot-check schema drift landed (sanity check before Bronze ingestion)

In [ ]:
sample_df = spark.read.json(f"{LOGS_PATH}/*/*/*.json")
sample_df.printSchema()
print(f"Total events readable by Spark: {sample_df.count():,}")
display(sample_df.limit(5))